# Dataset rescue statuses

Checks the rescue status of the [example data sources](https://docs.google.com/document/d/1rYsP1I1-TAd3Tu-G6EJ-5o7IbQIHf_dKT3DjH8IpQuo/edit?tab=t.0#heading=h.87di2doy0z55). Only includes the official government data sources that don't require auth.

Set up the path:


In [1]:
import sys
from pathlib import Path

repo_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "ptps_wildfire_demo").is_dir()
)
sys.path.insert(0, str(repo_root))

In [2]:
datasets = [
    {
        "name": "USFS probabilistic wildfire risk burn probability",
        "webpage": "https://data-usfs.hub.arcgis.com/datasets/usfs::probabilistic-wildfire-risk-burn-probability-image-service/explore",
        "example_data_url": "https://imagery.geoplatform.gov/iipp/rest/services/Fire_Aviation/USFS_EDW_RMRS_ProbabilisticWildfireRiskBurnProbability/ImageServer/exportImage?bbox=-2.00375070672E7,2135965.179399997,-7130157.067200001,1.1597575179399997E7&f=image",
    },
    {
        "name": "Wildfire Risk to Communities",
        "webpage": "https://wildfirerisk.org/download/",
        "example_data_url": "https://wildfirerisk.org/wp-content/uploads/2026/04/wrc_download_20260415.xlsx",
    },
    {
        "name": "CarbonPlan Open Climate Risk buildings",
        "webpage": "https://source.coop/carbonplan/carbonplan-ocr",
        "example_data_url": "https://s3.us-west-2.amazonaws.com/us-west-2.opendata.source.coop/carbonplan/carbonplan-ocr/output/fire-risk/vector/production/v1.1.0/geoparquet/buildings.parquet",
    },
    {
        "name": "NOAA weather alerts",
        "webpage": "https://www.weather.gov/documentation/services-web-api",
        "example_data_url": "https://api.weather.gov/alerts/active?event=Red%20Flag%20Warning",
    },
    {
        "name": "NOAA HMS fire and smoke product",
        "webpage": "https://www.ospo.noaa.gov/products/land/hms.html",
        "example_data_url": "https://satepsanone.nesdis.noaa.gov/pub/FIRE/web/HMS/Smoke_Polygons/KML/2026/07/hms_smoke20260701.kml",
    },
    {
        "name": "NOAA Storm Events Database",
        "webpage": "https://www.ncdc.noaa.gov/stormevents/",
        "example_data_url": "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_details-ftp_v1.0_d2025_c20260819.csv.gz",
    },
    {
        "name": "NASA FIRMS active fire detections",
        "webpage": "https://firms.modaps.eosdis.nasa.gov/",
        "example_data_url": "https://firms.modaps.eosdis.nasa.gov/data/active_fire/modis-c6.1/shapes/zips/MODIS_C6_1_USA_contiguous_and_Hawaii_24h.zip",
    },
    {
        "name": "WFIGS current interagency fire perimeters",
        "webpage": "https://wfigs-nifc.hub.arcgis.com/",
        "example_data_url": "https://gis.blm.gov/arcgis/rest/services/fire/BLM_Natl_FirePerimeter/MapServer/generateKml",
    },
    {
        "name": "MODIS NDVI granules",
        "webpage": "https://modis.gsfc.nasa.gov/data/dataprod/mod13.php",
        "example_data_url": "https://cmr.earthdata.nasa.gov/search/granules.json?short_name=MOD13Q1&page_size=1",
    },
    # requires POST
    # {
    #     "name": "OpenStreetMap infrastructure",
    #     "webpage": "https://overpass-api.de/",
    #     "example_data_url": "https://overpass-api.de/api/interpreter?data=%5Bout%3Ajson%5D%3Bway%5Bhighway%5D%2845.4%2C-122.8%2C45.5%2C-122.6%29%3Bout%20geom%3B",
    # },
    {
        "name": "USDA LANDFIRE seasonal fuels",
        "webpage": "https://www.landfire.gov/fuel/seasonal_fuels",
        "example_data_url": "https://www.landfire.gov/data-downloads/SeasonalFuels/LF2025_FBFM40_SU26.zip",
    },
    {
        "name": "CDC PLACES",
        "webpage": "https://www.cdc.gov/places/",
        "example_data_url": "https://data.cdc.gov/resource/i46a-9kgh.geojson",
    },
    {
        "name": "CDC/ATSDR Social Vulnerability Index",
        "webpage": "https://www.atsdr.cdc.gov/place-health/php/svi/index.html",
        "example_data_url": "https://svi2.cdc.gov/webapi/Documents/download?year=2022&type=csv&category=states_counties&name=SVI_2022_US_COUNTY",
    },
    # {
    #     "name": "EPA EJScreen",
    #     "webpage": "https://www.epa.gov/ejscreen",
    #     "example_data_url": "",
    # },
]


In [3]:
import asyncio

import httpx
import pandas as pd

from ptps_wildfire_demo.proxy.resolver import Resolver


async def get_rescues() -> list[dict[str, str | None]]:
    timeout = httpx.Timeout(30.0, connect=30.0)
    async with httpx.AsyncClient(timeout=timeout) as client:
        resolver = Resolver(client)
        rescues = await asyncio.gather(
            *(resolver.get_rescue(dataset["example_data_url"]) for dataset in datasets)
        )

    return [
        {
            "name": dataset["name"],
            "example_data_url": dataset["example_data_url"],
            "wayback_newest_url": rescue.wayback_newest_url,
            "drp_metadata_url": rescue.drp_metadata_url,
            "drp_download_location": rescue.drp_download_location,
        }
        for dataset, rescue in zip(datasets, rescues, strict=True)
    ]


results = pd.DataFrame(await get_rescues())
pd.set_option("display.max_colwidth", 200)
results

,name,example_data_url,wayback_newest_url,drp_metadata_url,drp_download_location
0,USFS probabilistic wildfire risk burn probability,"https://imagery.geoplatform.gov/iipp/rest/services/Fire_Aviation/USFS_EDW_RMRS_ProbabilisticWildfireRiskBurnProbability/ImageServer/exportImage?bbox=-2.00375070672E7,2135965.179399997,-7130157.067...",NaN,None,None
1,Wildfire Risk to Communities,https://wildfirerisk.org/wp-content/uploads/2026/04/wrc_download_20260415.xlsx,NaN,None,None
2,CarbonPlan Open Climate Risk buildings,https://s3.us-west-2.amazonaws.com/us-west-2.opendata.source.coop/carbonplan/carbonplan-ocr/output/fire-risk/vector/production/v1.1.0/geoparquet/buildings.parquet,NaN,None,None
3,NOAA weather alerts,https://api.weather.gov/alerts/active?event=Red%20Flag%20Warning,NaN,None,None
4,NOAA HMS fire and smoke product,https://satepsanone.nesdis.noaa.gov/pub/FIRE/web/HMS/Smoke_Polygons/KML/2026/07/hms_smoke20260701.kml,NaN,None,None
5,NOAA Storm Events Database,https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_details-ftp_v1.0_d2025_c20260819.csv.gz,NaN,None,None
6,NASA FIRMS active fire detections,https://firms.modaps.eosdis.nasa.gov/data/active_fire/modis-c6.1/shapes/zips/MODIS_C6_1_USA_contiguous_and_Hawaii_24h.zip,http://web.archive.org/web/20260110052750/https://firms.modaps.eosdis.nasa.gov/data/active_fire/modis-c6.1/shapes/zips/MODIS_C6_1_USA_contiguous_and_Hawaii_24h.zip,None,None
7,WFIGS current interagency fire perimeters,https://gis.blm.gov/arcgis/rest/services/fire/BLM_Natl_FirePerimeter/MapServer/generateKml,http://web.archive.org/web/20260209203352/https://gis.blm.gov/arcgis/rest/services/fire/BLM_Natl_FirePerimeter/MapServer/generateKml,None,None
8,MODIS NDVI granules,https://cmr.earthdata.nasa.gov/search/granules.json?short_name=MOD13Q1&page_size=1,NaN,None,None
9,USDA LANDFIRE seasonal fuels,https://www.landfire.gov/data-downloads/SeasonalFuels/LF2025_FBFM40_SU26.zip,NaN,None,None
